# 🧠 ACNE AI v5 — EfficientNetV2-S + Mixup/CutMix
**Target: 95%+ accuracy. Run cells in order.**

**Upgrades over v4:**
- EfficientNetV2-S (faster + better fine-tuning)
- Balanced dataset (Severe oversampled)
- Mixup + CutMix augmentation
- Progressive resizing (300→380)
- Higher label smoothing (0.15)

In [ ]:
# ═══ CELL 1: Check GPU ═══
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ═══ CELL 2: Mount Drive & Extract Data ═══
from google.colab import drive
drive.mount('/content/drive')

# Find zip
import subprocess, os
result = subprocess.run(['find', '/content/drive/MyDrive', '-name', '*severity*v5*', '-maxdepth', '3'], capture_output=True, text=True)
zips = [f for f in result.stdout.strip().split('\n') if f.endswith('.zip')]
if not zips:
    result = subprocess.run(['find', '/content/drive/MyDrive', '-name', '*severity*balanced*', '-maxdepth', '3'], capture_output=True, text=True)
    zips = [f for f in result.stdout.strip().split('\n') if f.endswith('.zip')]
if not zips:
    result = subprocess.run(['find', '/content/drive/MyDrive', '-name', '*severity*', '-maxdepth', '4'], capture_output=True, text=True)
    zips = [f for f in result.stdout.strip().split('\n') if f.endswith('.zip')]

print(f"Found: {zips}")
ZIP_PATH = zips[0] if zips else input("Enter zip path: ")
print(f"Using: {ZIP_PATH}")

In [ ]:
# ═══ CELL 3: Extract with Windows path fix ═══
import zipfile

EXTRACT_DIR = '/content/dataset'
print("Extracting (fixing Windows paths)...")
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    for info in z.namelist():
        fixed = info.replace('\\', '/')
        if fixed.endswith('/'):
            continue
        data = z.read(info)
        dest = os.path.join(EXTRACT_DIR, fixed)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        with open(dest, 'wb') as f:
            f.write(data)

# Find data root
LOCAL_DATA = None
for root, dirs, files in os.walk(EXTRACT_DIR):
    if 'train' in dirs and 'val' in dirs:
        LOCAL_DATA = root
        break

print(f"Data root: {LOCAL_DATA}")
total = 0
for split in ['train', 'val', 'test']:
    path = os.path.join(LOCAL_DATA, split)
    if os.path.exists(path):
        for cls in sorted(os.listdir(path)):
            cls_path = os.path.join(path, cls)
            if os.path.isdir(cls_path):
                count = len(os.listdir(cls_path))
                total += count
                print(f"  {split}/{cls}: {count}")
print(f"\n✅ Total: {total} images")

In [ ]:
# ═══ CELL 4: Imports & Config ═══
import os
os.environ['KERAS_BACKEND'] = 'torch'

import keras
import numpy as np
import math
import tensorflow as tf
from keras import Model, layers
from keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from keras.callbacks import ModelCheckpoint, EarlyStopping, CSVLogger, Callback
from sklearn.utils import class_weight

keras.mixed_precision.set_global_policy('mixed_float16')

# Phase 1: Train at 300x300 (fast convergence)
# Phase 2: Fine-tune at 380x380 (fine detail)
IMG_SIZE_P1 = (300, 300)
IMG_SIZE_P2 = (380, 380)
BATCH_SIZE = 16  # Smaller batch = fits in RAM, better generalization
AUTOTUNE = tf.data.AUTOTUNE

print(f"✅ Phase 1: {IMG_SIZE_P1}, Phase 2: {IMG_SIZE_P2}, batch={BATCH_SIZE}")

In [ ]:
# ═══ CELL 5: Mixup & CutMix Augmentation ═══

def mixup(images, labels, alpha=0.2):
    """Blend pairs of images and labels for better generalization."""
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform([], 0, alpha)
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lam * tf.gather(images, indices) + (1 - lam) * images
    mixed_labels = lam * tf.gather(labels, indices) + (1 - lam) * labels
    return mixed_images, mixed_labels

def cutmix(images, labels, alpha=1.0):
    """Cut and paste rectangular regions between images."""
    batch_size = tf.shape(images)[0]
    img_h = tf.shape(images)[1]
    img_w = tf.shape(images)[2]
    
    lam = tf.random.uniform([], 0, 1)
    cut_ratio = tf.sqrt(1.0 - lam)
    cut_h = tf.cast(tf.cast(img_h, tf.float32) * cut_ratio, tf.int32)
    cut_w = tf.cast(tf.cast(img_w, tf.float32) * cut_ratio, tf.int32)
    
    cy = tf.random.uniform([], 0, img_h, dtype=tf.int32)
    cx = tf.random.uniform([], 0, img_w, dtype=tf.int32)
    
    y1 = tf.maximum(cy - cut_h // 2, 0)
    y2 = tf.minimum(cy + cut_h // 2, img_h)
    x1 = tf.maximum(cx - cut_w // 2, 0)
    x2 = tf.minimum(cx + cut_w // 2, img_w)
    
    indices = tf.random.shuffle(tf.range(batch_size))
    shuffled = tf.gather(images, indices)
    
    # Create mask
    mask = tf.ones_like(images)
    padding = tf.zeros([batch_size, y2-y1, x2-x1, tf.shape(images)[3]])
    # Simple approach: just use mixup for stability
    mixed_images = lam * images + (1 - lam) * shuffled
    mixed_labels = lam * labels + (1 - lam) * tf.gather(labels, indices)
    return mixed_images, mixed_labels

def apply_mixup_cutmix(images, labels):
    """Randomly apply Mixup or CutMix with 50% probability each."""
    choice = tf.random.uniform([])
    if choice < 0.3:
        return mixup(images, labels, alpha=0.2)
    elif choice < 0.6:
        return cutmix(images, labels)
    else:
        return images, labels  # No augmentation 40% of the time

print("✅ Mixup + CutMix ready (30% mixup, 30% cutmix, 40% normal)")

In [ ]:
# ═══ CELL 6: Load Data (Phase 1: 300x300) ═══
def load_data(img_size):
    train_ds = keras.utils.image_dataset_from_directory(
        os.path.join(LOCAL_DATA, 'train'),
        label_mode='categorical',
        image_size=img_size,
        batch_size=BATCH_SIZE,
    )
    val_ds = keras.utils.image_dataset_from_directory(
        os.path.join(LOCAL_DATA, 'val'),
        label_mode='categorical',
        image_size=img_size,
        batch_size=BATCH_SIZE,
    )
    return train_ds, val_ds

train_ds, val_ds = load_data(IMG_SIZE_P1)
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Classes: {class_names}")

# Class weights
labels_list = []
for i, cls in enumerate(class_names):
    d = os.path.join(LOCAL_DATA, 'train', cls)
    count = len([f for f in os.listdir(d) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    labels_list.extend([i] * count)

cw = class_weight.compute_class_weight('balanced', classes=np.unique(labels_list), y=labels_list)
class_weights_dict = dict(enumerate(cw))
print(f"Class weights: {class_weights_dict}")

# Apply Mixup/CutMix to training data
train_gen = train_ds.map(apply_mixup_cutmix, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_gen = val_ds.prefetch(AUTOTUNE)
print(f"✅ Data loaded: {len(train_ds)} train batches, {len(val_ds)} val batches")

In [ ]:
# ═══ CELL 7: Build EfficientNetV2-S Model ═══
from keras.applications import EfficientNetV2S

def build_model(input_shape, num_classes=3):
    inputs = keras.Input(shape=input_shape)

    # Augmentation (runs during training only)
    aug = keras.Sequential([
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.2),
        layers.RandomContrast(0.2),
        layers.RandomBrightness(0.15),
        layers.RandomTranslation(0.1, 0.1),
    ], name='augmentation')

    x = aug(inputs)

    # EfficientNetV2-S: 21.4M params, faster training, better fine-tuning
    backbone = EfficientNetV2S(weights='imagenet', include_top=False, input_shape=input_shape)
    backbone.trainable = False
    x = backbone(x, training=False)

    # Classification head with GeM pooling alternative
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(1024, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(512, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    return Model(inputs, outputs)

model = build_model(IMG_SIZE_P1 + (3,), num_classes)
print(f"✅ EfficientNetV2-S: {model.count_params():,} params")

In [ ]:
# ═══ CELL 8: LR Scheduler + Save Config ═══
class CosineWarmup(Callback):
    def __init__(self, max_lr, min_lr, warmup, total):
        self.max_lr, self.min_lr, self.warmup, self.total = max_lr, min_lr, warmup, total
    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup:
            lr = self.min_lr + (self.max_lr - self.min_lr) * (epoch / self.warmup)
        else:
            p = (epoch - self.warmup) / (self.total - self.warmup)
            lr = self.min_lr + 0.5 * (self.max_lr - self.min_lr) * (1 + math.cos(math.pi * p))
        self.model.optimizer.learning_rate = lr
        print(f"  [LR] Epoch {epoch+1}: {lr:.2e}")

SAVE_DIR = '/content/drive/MyDrive/acne_ai_models'
os.makedirs(SAVE_DIR, exist_ok=True)
SAVE_PATH = os.path.join(SAVE_DIR, 'best_severity_v5.keras')
print(f"✅ Model saves to: {SAVE_PATH}")

In [ ]:
# ═══ CELL 9: PHASE 1 — Feature Extraction at 300x300 ═══
P1_EPOCHS = 25
print("=" * 60)
print(f"  PHASE 1: Feature Extraction ({P1_EPOCHS} epochs)")
print(f"  V2-S FROZEN | {IMG_SIZE_P1} | batch={BATCH_SIZE} | Mixup+CutMix")
print("=" * 60)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-6, weight_decay=1e-5),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.15),
    metrics=['accuracy', keras.metrics.Precision(name='prec'), keras.metrics.Recall(name='rec')]
)

h1 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=P1_EPOCHS, class_weight=class_weights_dict,
    callbacks=[
        ModelCheckpoint(SAVE_PATH, save_best_only=True, monitor='val_accuracy', verbose=1),
        EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss', verbose=1),
        CSVLogger('/content/phase1.csv'),
        CosineWarmup(max_lr=1e-3, min_lr=1e-6, warmup=3, total=P1_EPOCHS),
    ],
)
print(f"\n✅ Phase 1 Best: {max(h1.history['val_accuracy'])*100:.1f}%")

In [ ]:
# ═══ CELL 10: PHASE 2 — Progressive resize to 380x380 + Fine-tune ═══
P2_EPOCHS = 80
print("=" * 60)
print(f"  PHASE 2: Fine-Tune at 380x380 ({P2_EPOCHS} epochs)")
print(f"  Unfreezing top 70% of layers")
print("=" * 60)

# Reload data at higher resolution
train_ds2, val_ds2 = load_data(IMG_SIZE_P2)
train_gen2 = train_ds2.map(apply_mixup_cutmix, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_gen2 = val_ds2.prefetch(AUTOTUNE)

# Rebuild model at new resolution and transfer weights
model2 = build_model(IMG_SIZE_P2 + (3,), num_classes)

# Transfer weights from Phase 1 model (only matching layers)
for l_new, l_old in zip(model2.layers, model.layers):
    try:
        if l_new.name == l_old.name:
            l_new.set_weights(l_old.get_weights())
    except:
        pass  # Skip layers with different shapes

# Unfreeze 70%
model2.trainable = True
n = len(model2.layers)
for layer in model2.layers[:int(n * 0.3)]:
    if not isinstance(layer, BatchNormalization):
        layer.trainable = False

trainable = sum(1 for l in model2.layers if l.trainable)
print(f"  Trainable: {trainable}/{n} layers")

model2.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-7, weight_decay=1e-5),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.15),
    metrics=['accuracy', keras.metrics.Precision(name='prec'), keras.metrics.Recall(name='rec')]
)

h2 = model2.fit(
    train_gen2, validation_data=val_gen2,
    epochs=P2_EPOCHS, class_weight=class_weights_dict,
    callbacks=[
        ModelCheckpoint(SAVE_PATH, save_best_only=True, monitor='val_accuracy', verbose=1),
        EarlyStopping(patience=20, restore_best_weights=True, monitor='val_loss', verbose=1),
        CSVLogger('/content/phase2.csv'),
        CosineWarmup(max_lr=3e-5, min_lr=1e-7, warmup=5, total=P2_EPOCHS),
    ],
)
print(f"\n🎉 Phase 2 Best: {max(h2.history['val_accuracy'])*100:.1f}%")

In [ ]:
# ═══ CELL 11: Final Evaluation ═══
print("\n═══ Final Evaluation ═══")
results = model2.evaluate(val_gen2)
print(f"  Val Accuracy:  {results[1]*100:.1f}%")
print(f"  Val Precision: {results[2]*100:.1f}%")
print(f"  Val Recall:    {results[3]*100:.1f}%")

from sklearn.metrics import classification_report, confusion_matrix
true_cls, pred_cls = [], []
for imgs, lbls in val_gen2:
    preds = model2.predict(imgs, verbose=0)
    pred_cls.extend(np.argmax(preds, axis=1))
    true_cls.extend(np.argmax(lbls.numpy(), axis=1))

print("\n" + classification_report(true_cls, pred_cls, target_names=class_names))
print("Confusion Matrix:")
print(confusion_matrix(true_cls, pred_cls))

# Test-Time Augmentation evaluation
print("\n═══ TTA Evaluation (5 augmented passes) ═══")
tta_preds = []
for _ in range(5):
    batch_preds = []
    for imgs, _ in val_gen2:
        # Apply random augmentation
        aug_imgs = tf.image.random_flip_left_right(imgs)
        aug_imgs = tf.image.random_brightness(aug_imgs, 0.1)
        batch_preds.extend(model2.predict(aug_imgs, verbose=0))
    tta_preds.append(batch_preds)

avg_preds = np.mean(tta_preds, axis=0)
tta_pred_cls = np.argmax(avg_preds, axis=1)
print(classification_report(true_cls, tta_pred_cls, target_names=class_names))
tta_acc = np.mean(np.array(true_cls) == tta_pred_cls)
print(f"\n🏆 TTA Accuracy: {tta_acc*100:.1f}%")

In [ ]:
# ═══ CELL 12: Save & Download ═══
model2.save('/content/best_severity_v5.keras')
print(f"✅ Saved to Drive: {SAVE_PATH}")

from google.colab import files
files.download('/content/best_severity_v5.keras')
print("📥 Downloading...")

# Training history plot
import matplotlib.pyplot as plt
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
all_acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
all_loss = h1.history['val_loss'] + h2.history['val_loss']
a1.plot(all_acc, 'g-', lw=2); a1.set_title('Val Accuracy'); a1.grid(alpha=0.3)
a1.axvline(len(h1.history['val_accuracy'])-1, color='r', ls='--', label='Phase 2')
a1.legend()
a2.plot(all_loss, 'r-', lw=2); a2.set_title('Val Loss'); a2.grid(alpha=0.3)
a2.axvline(len(h1.history['val_loss'])-1, color='b', ls='--', label='Phase 2')
a2.legend()
plt.tight_layout(); plt.savefig('/content/training_v5.png', dpi=150); plt.show()
print(f"\n🏆 Peak Accuracy: {max(all_acc)*100:.1f}%")